# Explore reprompt history
Run these cells in order to browse revision instructions, input drafts, revised responses, and saved ticket messages. Change the database path and filters as needed. Rerun the load cell to refresh the data.

Database access is read-only; no backend imports or model calls. Requires a Python Jupyter kernel with IPython, but not pandas. Older records may lack input drafts or message snapshots.

By default this reads the live Docker Compose `fastapi` database. Docker must be running and accessible to the notebook kernel. Set `DATA_SOURCE = "local"` to inspect `DB_PATH` instead. The local file and Docker volume are separate databases.


In [ ]:
import json
import os
import sqlite3
import subprocess
from pathlib import Path
from contextlib import closing
from html import escape
from IPython.display import HTML, display

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "backend" / "database.py").is_file()), None)
if root is None:
    raise RuntimeError("Launch JupyterLab from inside the project directory.")
# Use "docker" for the running Compose backend, or "local" for a file.
DATA_SOURCE = "docker"

# Override for another database or a deployment snapshot.
# Reads process environment, not .env. Relative paths use the kernel cwd.
DB_PATH = Path(os.environ.get("CACHE_DATABASE_PATH", root / "backend" / "cache.db")).expanduser().resolve()
print(f"Source: {DATA_SOURCE}; local database option: {DB_PATH}")


In [ ]:
# Rerun this cell after reprompting to refresh the history.
def read_events(conn):
    conn.row_factory = sqlite3.Row
    columns = {r["name"] for r in conn.execute("PRAGMA table_info(generation_logs)")}
    if not columns:
        raise RuntimeError("No generation_logs table. Select the backend cache database.")
    order = "created_at DESC, id DESC" if "id" in columns else "created_at DESC"
    return [dict(r) for r in conn.execute(
        f"SELECT * FROM generation_logs WHERE event_type = ? ORDER BY {order}", ("reprompt",))]

if DATA_SOURCE == "docker":
    # Query in read-only mode inside the active service; no copying a live SQLite file.
    script = """
import json, os, sqlite3
from pathlib import Path
path = Path(os.environ.get("CACHE_DATABASE_PATH", "/app/cache.db")).resolve()
conn = sqlite3.connect(path.as_uri() + "?mode=ro", uri=True)
conn.row_factory = sqlite3.Row
try:
    rows = [dict(r) for r in conn.execute(
        "SELECT * FROM generation_logs WHERE event_type = ? ORDER BY created_at DESC, id DESC", ("reprompt",))]
    print(json.dumps({"database": str(path), "events": rows}))
finally:
    conn.close()
"""
    try:
        result = subprocess.run(
            ["docker", "compose", "exec", "-T", "fastapi", "python", "-c", script],
            cwd=root, capture_output=True, text=True, encoding="utf-8", timeout=30,
        )
    except (OSError, subprocess.TimeoutExpired) as exc:
        raise RuntimeError("Cannot query Docker. Start Docker and the fastapi service, or choose DATA_SOURCE = 'local'.") from exc
    if result.returncode:
        raise RuntimeError(f"Docker query failed. Check the fastapi service and Docker access.\n{result.stderr}")
    payload = json.loads(result.stdout)
    events = payload["events"]
    print(f"Docker database: {payload['database']}")
elif DATA_SOURCE == "local":
    if not DB_PATH.is_file():
        raise FileNotFoundError(DB_PATH)
    with closing(sqlite3.connect(DB_PATH.as_uri() + "?mode=ro", uri=True)) as conn:
        events = read_events(conn)
    print(f"Local database: {DB_PATH}")
else:
    raise ValueError("DATA_SOURCE must be 'docker' or 'local'.")
print(f"Loaded {len(events)} reprompts across {len({e['ticket_id'] for e in events})} tickets.")


## Filter history
Leave filters empty to include everything. Dates are inclusive `YYYY-MM-DD`, using the stored timestamp date. Search covers instructions, input drafts, revisions, and ticket messages.


In [ ]:
TICKET_ID = ""
SEARCH = ""
DATE_FROM = ""
DATE_TO = ""
PAGE = 1
PAGE_SIZE = 25

filtered = [e for e in events if
    (not TICKET_ID or str(e["ticket_id"]) == str(TICKET_ID))
    and (not DATE_FROM or e["created_at"][:10] >= DATE_FROM)
    and (not DATE_TO or e["created_at"][:10] <= DATE_TO)
    and (not SEARCH or SEARCH.casefold() in " ".join(str(e.get(k) or "") for k in
         ("instructions", "input_response_text", "response_text", "messages_json")).casefold())]
if PAGE < 1 or PAGE_SIZE < 1:
    raise ValueError("PAGE and PAGE_SIZE must be positive.")
start = (PAGE - 1) * PAGE_SIZE
rows = []
for index, e in enumerate(filtered[start:start + PAGE_SIZE], start):
    values = [index, e.get("id", "legacy"), e["ticket_id"], e["created_at"], e.get("instructions", "")]
    rows.append("<tr>" + "".join("<td style='vertical-align:top;white-space:pre-wrap'>" + escape(str(v)) + "</td>" for v in values) + "</tr>")
print(f"{len(filtered)} matches; page {PAGE} of {max(1, (len(filtered) + PAGE_SIZE - 1) // PAGE_SIZE)}")
display(HTML("<table><tr><th>Index</th><th>Event ID</th><th>Ticket</th><th>Created</th><th>Instructions</th></tr>" + "".join(rows) + "</table>"))


## Inspect a revision
Choose an **Index** from the filtered table. The input response is the exact draft saved for this revision, which may itself be an earlier revision.


In [ ]:
SELECTED_INDEX = 0

def panel(title, value):
    text = "Not captured in this historical record." if value is None else str(value)
    return f"<h3>{escape(title)}</h3><pre style='white-space:pre-wrap;overflow-wrap:anywhere'>{escape(text)}</pre>"

if not filtered:
    print("No matching reprompts. Change filters or select another database.")
elif not 0 <= SELECTED_INDEX < len(filtered):
    print(f"Choose an index from 0 to {len(filtered) - 1}.")
else:
    selected = filtered[SELECTED_INDEX]
    print(f"Ticket {selected['ticket_id']} | Event {selected.get('id', 'legacy')} | {selected['created_at']}")
    print(f"Based on message: {selected['based_on_message_id']} | First draft ID: {selected.get('first_draft_id')} | Prompt version: {selected.get('prompt_version', '')}")
    display(HTML(panel("Revision instructions", selected.get("instructions"))))
    display(HTML("<div style='display:flex;gap:24px;flex-wrap:wrap'>"
        + "<div style='flex:1;min-width:280px'>" + panel("Input response", selected.get("input_response_text")) + "</div>"
        + "<div style='flex:1;min-width:280px'>" + panel("Revised response", selected.get("response_text")) + "</div></div>"))
    for title, key in [("Ticket messages at revision time", "messages_json"), ("Saved sources", "sources_json")]:
        raw = selected.get(key)
        try:
            value = json.dumps(json.loads(raw), ensure_ascii=False, indent=2) if raw is not None else None
        except (TypeError, ValueError):
            value = raw
        display(HTML(panel(title, value)))


Ticket messages are the snapshot logged by the endpoint. The current `reprompt_call()` receives the instructions and input draft, not this snapshot. Saved sources are carried over by the endpoint, not a separate trace of revision tool calls.

Outputs are empty in the committed notebook. Clear outputs before sharing if you run it against customer data.
